# Complete Project Demo - Multi-Channel Epileptic Seizure Prediction

This single notebook combines the four demo notebooks into one walkthrough for recording or presenting the project.

It covers:

1. Environment and artifact checks
2. Dataset and preprocessing summary
3. Sharded data pipeline overview
4. Model/training pipeline overview
5. Evaluation results and plots
6. Label audit and preictal horizon ablation
7. Explainability status and paper package review

This notebook is demo-safe: it displays existing validated artifacts and does **not** rerun preprocessing, full training, or evaluation.


## Installation Needed

Recommended from the repository root:

```powershell
cd D:\epilepsy\seizure-prediction
.\.venv\Scripts\activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name seizure-prediction --display-name "Seizure Prediction (.venv)"
jupyter lab
```

For a lighter demo-only environment:

```powershell
python -m pip install -r notebooks\requirements_demo.txt
```

Use the existing project `.venv` when possible because it already contains the tested PyTorch/CUDA stack.


## 1. Project Setup

This cell makes the notebook runnable from either the repository root or the `notebooks/` folder.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PAPER_DIR = PROJECT_ROOT / 'paper'
TABLE_DIR = PAPER_DIR / 'tables'
FIGURE_DIR = PAPER_DIR / 'figures'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

print('Project root:', PROJECT_ROOT)
print('Paper directory:', PAPER_DIR)
print('Tables directory exists:', TABLE_DIR.exists())
print('Figures directory exists:', FIGURE_DIR.exists())


## 2. Helper Functions

These helpers keep the demo clean by checking whether each artifact exists before displaying it.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, Markdown


def load_csv(name, max_rows=None):
    path = TABLE_DIR / name
    if not path.exists():
        print(f'MISSING: {path}')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'{name}: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head(max_rows) if max_rows else df)
    return df


def show_figure(name, width=760):
    path = FIGURE_DIR / name
    if not path.exists():
        print(f'MISSING: {path}')
        return
    print(f'{name} ({path.stat().st_size:,} bytes)')
    display(Image(filename=str(path), width=width))


def show_markdown_file(path, max_chars=4000):
    path = Path(path)
    if not path.exists():
        print(f'MISSING: {path}')
        return
    text = path.read_text(encoding='utf-8', errors='replace')[:max_chars]
    display(Markdown(text))


## 3. Dataset and Preprocessing Summary

The project uses CHB-MIT EDF shards with 18 common EEG channels, 0.5-40 Hz bandpass filtering, a 50 Hz notch filter, 4-second windows, 2-second stride, per-window z-score normalization, a 10-minute preictal horizon in the baseline, and patient-wise splitting.


In [ ]:
dataset_summary = load_csv('dataset_summary.csv')
patient_split = load_csv('patient_split_table.csv')
class_distribution = load_csv('class_distribution_table.csv')


### Preprocessing and Data Flow Figures


In [ ]:
for fig in [
    'pipeline_diagram.png',
    'data_flow_diagram.png',
    'preprocessing_flowchart.png',
    'patient_split_diagram.png',
    'positive_negative_distribution.png',
    'dataset_statistics_by_patient.png',
]:
    show_figure(fig)


## 4. Sharded Dataset Check

The training pipeline is designed around per-EDF shard files under `data/interim/windows/`. This quick check counts available local shards without loading them into RAM.


In [ ]:
WINDOW_DIR = PROJECT_ROOT / 'data' / 'interim' / 'windows'
if WINDOW_DIR.exists():
    x_files = sorted(WINDOW_DIR.glob('*_X.npy'))
    y_files = sorted(WINDOW_DIR.glob('*_y.npy'))
    meta_files = sorted(WINDOW_DIR.glob('*_meta.csv'))
    print('Shard directory:', WINDOW_DIR)
    print('X shards:', len(x_files))
    print('y shards:', len(y_files))
    print('metadata shards:', len(meta_files))
    print('First shards:', [p.name for p in x_files[:5]])
else:
    print('Shard directory is not present in this checkout:', WINDOW_DIR)
    print('This is expected if raw/interim data are not included in GitHub.')


## 5. Model and Training Configuration

This section displays the frozen model and training configuration used by the completed experiments.


In [ ]:
model_config = load_csv('model_configuration_table.csv')
training_config = load_csv('training_configuration_table.csv')
hyperparameters = load_csv('hyperparameter_table.csv')
software = load_csv('software_environment_table.csv')
hardware = load_csv('hardware_configuration_table.csv')


### Instantiate All Three Model Families

This verifies that the model-building code imports correctly and gives a demo-friendly parameter count. It does not train the models.


In [ ]:
import torch
from src.training.train import build_model

model_names = ['eegnet', 'attention', 'hybrid']
demo_input = torch.zeros(2, 1, 18, 1024)  # batch=2, channels=18, samples=1024

for model_name in model_names:
    model = build_model(model_name)
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        output = model(demo_input)
    print(f'{model_name.upper():9s} | parameters: {params:,} | output shape: {tuple(output.shape)}')


### Training Pipeline and Curves


In [ ]:
for fig in [
    'training_pipeline.png',
    'learning_curves_eegnet.png',
    'validation_curves_eegnet.png',
    'model_comparison_chart.png',
]:
    show_figure(fig)


## 6. Evaluation Results

All values shown here come from existing CSV/JSON artifacts in the paper package.


In [ ]:
model_comparison = load_csv('model_comparison_table.csv')
optimization_summary = load_csv('optimization_summary_table.csv')


### ROC, Precision-Recall, and Confusion Matrix


In [ ]:
for fig in [
    'roc_curve_eegnet.png',
    'pr_curve_eegnet.png',
    'confusion_matrix_eegnet.png',
]:
    show_figure(fig)


## 7. Label Audit and Preictal Horizon Ablation

This section summarizes the scientific audit of label quality and the controlled preictal horizon ablation.


In [ ]:
label_audit = load_csv('label_audit_summary_table.csv')
horizon_stats = load_csv('preictal_horizon_dataset_statistics.csv')
horizon_ablation = load_csv('preictal_horizon_ablation_table.csv')


In [ ]:
for fig in [
    'distance_to_seizure_histogram.png',
    'preictal_horizon_pr_auc.png',
    'preictal_horizon_f1.png',
    'positive_class_size_vs_horizon.png',
]:
    show_figure(fig)


## 8. Explainability and SHAP Status

The repository contains SHAP/explainability code, but the final paper package does not claim validated SHAP results unless corresponding measured artifacts exist. This section is useful for honestly explaining what is implemented versus what is included in the frozen results.


In [ ]:
explainability_dir = PROJECT_ROOT / 'src' / 'explainability'
print('Explainability directory:', explainability_dir)
if explainability_dir.exists():
    for path in sorted(explainability_dir.glob('*.py')):
        print('-', path.relative_to(PROJECT_ROOT))
else:
    print('No explainability source directory found.')

shap_like = sorted(PAPER_DIR.rglob('*shap*')) + sorted((PROJECT_ROOT / 'outputs').rglob('*shap*')) if (PROJECT_ROOT / 'outputs').exists() else sorted(PAPER_DIR.rglob('*shap*'))
print('SHAP-like paper/output artifacts found:', len(shap_like))
for path in shap_like[:20]:
    print('-', path.relative_to(PROJECT_ROOT))


## 9. Paper Package and Validation Reports

This section shows the final package inventory and validation report used for submission preparation.


In [ ]:
manifest = load_csv('paper_package_manifest.csv', max_rows=20)
master_index = load_csv('master_experiment_index.csv', max_rows=20)


In [ ]:
show_markdown_file(PAPER_DIR / 'FINAL_VALIDATION_REPORT.md', max_chars=5000)


## 10. Optional Commands

The commands below are intentionally commented out. They are useful for explaining reproducibility, but they should not be run during a short demo video.


In [ ]:
# Heavy / frozen preprocessing command. Do not run during demo.
# !python src/data/preprocess_eeg.py

# Heavy / frozen experiment command. Do not run during demo.
# !python src/training/overnight_experiment_runner.py

# Optional future explainability command; not part of the frozen final results.
# !python src/explainability/shap_explainer.py --help


## Demo Closing Points

Key points to emphasize in the video:

- The data pipeline uses per-EDF shards to avoid loading the complete dataset into RAM.
- Patient-wise splitting is used to reduce leakage risk.
- The experiments include imbalance strategies, label audits, and a controlled preictal horizon ablation.
- The final claims are intentionally conservative and tied to measured artifacts.
- The notebook is for demonstration and reproducibility review, not for rerunning expensive experiments live.
